# NyayaSahayak - Legal Domain Classifier
CSE276 Project P14  
Team: Ravnoor Kaur (12505889) & Adarsh Kumar Shukla (12513184)  

This notebook trains a machine learning classifier to categorize legal queries and statutory sections into 5 domains:
- `criminal` (BNS / BNSS)
- `constitutional` (Constitution of India)
- `cyber` (Information Technology Act 2000)
- `consumer` (Consumer Protection Act 2019)
- `evidence` (Bharatiya Sakshya Adhiniyam 2023)

### 1. Load preprocessed legal chunks
We load the 2,419 statutory passages generated from the 6 indexed Indian acts.

In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the data directory whether running from root, backend, or notebooks/
for base in [Path('data'), Path('../data'), Path('backend/data'), Path('../../backend/data')]:
    if (base / 'processed' / 'chunks.jsonl').exists():
        data_dir = base
        break
else:
    raise FileNotFoundError('Could not find data/processed/chunks.jsonl')

data_file = data_dir / 'processed' / 'chunks.jsonl'
with open(data_file, 'r', encoding='utf-8') as f:
    rows = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(rows)
print(f'Total legal chunks loaded: {len(df):,}')
df[['act', 'section', 'title', 'category', 'year']].head()

Total legal chunks loaded: 2,419


,act,section,title,category,year
0,"Bharatiya Nyaya Sanhita, 2023",1,"Short title, commencement and application",criminal,2023
1,"Bharatiya Nyaya Sanhita, 2023",2,Definitions.,criminal,2023
2,"Bharatiya Nyaya Sanhita, 2023",3,General explanations,criminal,2023
3,"Bharatiya Nyaya Sanhita, 2023",4,Punishments,criminal,2023
4,"Bharatiya Nyaya Sanhita, 2023",5,Commutation of sentence,criminal,2023


### 2. Check class balance across legal categories
Criminal law has more sections by nature (BNS + BNSS + IPC mappings), so we will use class balancing during training.

In [2]:
counts = df['category'].value_counts().to_frame(name='Count')
counts['Percentage (%)'] = (counts['Count'] / len(df) * 100).round(2)
counts.index.name = 'Category'
counts

,Count,Percentage (%)
Category,,
criminal,1601,66.18
constitutional,493,20.38
evidence,198,8.19
consumer,84,3.47
cyber,43,1.78


### 3. Prepare features and split into Train/Test sets (80-20 stratified)
We combine the Act name, Section Title, and Section Content as text features, and perform a stratified 80-20 split so all 5 domains are proportionally represented.

In [3]:
from sklearn.model_selection import train_test_split

# Combine act, title and body text for rich feature representation
X = [f"{r.get('act', '')} {r.get('title', '')} {r.get('text', '')}" for r in rows]
y = [r['category'] for r in rows]

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train):,} (80%)')
print(f'Test size:  {len(X_test):,} (20%)')

Train size: 1,935 (80%)
Test size:  484 (20%)


### 4. Create and train pipeline (TF-IDF + Logistic Regression)
We build an end-to-end scikit-learn Pipeline with sublinear TF-IDF word unigrams and bigrams, and a balanced Multinomial Logistic Regression classifier.

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import time

clf = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
)

t0 = time.time()
clf.fit(X_train, y_train)
print(f'Pipeline training finished in {time.time() - t0:.2f} seconds.')

classes = list(clf.classes_)
print(f'Classes: {classes}')

vectorizer = clf.named_steps['tfidfvectorizer']
print(f'Vocabulary size: {len(vectorizer.vocabulary_):,} n-grams')

Pipeline training finished in 1.6 seconds.
Classes: [np.str_('constitutional'), np.str_('consumer'), np.str_('criminal'), np.str_('cyber'), np.str_('evidence')]
Vocabulary size: 21,873 n-grams


### 5. Evaluate on test set (Classification Report)
We evaluate precision, recall, and F1-score on the unseen 20% test partition (484 samples).

In [5]:
from sklearn.metrics import classification_report

y_pred = clf.predict(X_test)
report = classification_report(y_test, y_pred, digits=4, output_dict=True)
pd.DataFrame(report).transpose().round(4)

,precision,recall,f1-score,support
constitutional,0.9900,1.0000,0.9950,99.0000
consumer,1.0000,1.0000,1.0000,17.0000
criminal,1.0000,0.9969,0.9984,320.0000
cyber,1.0000,1.0000,1.0000,8.0000
evidence,1.0000,1.0000,1.0000,40.0000
accuracy,0.9979,0.9979,0.9979,0.9979
macro avg,0.9980,0.9994,0.9987,484.0000
weighted avg,0.9980,0.9979,0.9979,484.0000


### 6. Confusion Matrix
Evaluating actual vs predicted classifications across all 5 legal domains.

In [6]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
cm_df = pd.DataFrame(cm, index=clf.classes_, columns=clf.classes_)
cm_df.index.name = 'Actual Ground Truth'
cm_df.columns.name = 'Predicted'
cm_df

Predicted,constitutional,consumer,criminal,cyber,evidence
Actual Ground Truth,,,,,
constitutional,99,0,0,0,0
consumer,0,17,0,0,0
criminal,1,0,319,0,0
cyber,0,0,0,8,0
evidence,0,0,0,0,40


### 7. Test on sample user queries
Testing representative citizen questions across each of the 5 legal domains.

In [7]:
samples = [
    'Can police arrest without warrant in theft cases under BNSS?',
    'What are fundamental rights guaranteed under Article 21 of the Constitution?',
    'Consumer complaint against seller for defective goods and refund',
    'How to verify a digital signature certificate under Information Technology rules?',
    'Admissibility of electronic records and secondary evidence under Bharatiya Sakshya Adhiniyam'
]

for s in samples:
    probs = clf.predict_proba([s])[0]
    top_class = clf.classes_[np.argmax(probs)]
    conf = np.max(probs) * 100
    print(f"Query: '{s}'")
    print(f" -> Predicted: {top_class.upper()} ({conf:.1f}% confidence)\n")

Query: 'Can police arrest without warrant in theft cases under BNSS?'
 -> Predicted: CRIMINAL (59.7% confidence)

Query: 'What are fundamental rights guaranteed under Article 21 of the Constitution?'
 -> Predicted: CONSTITUTIONAL (71.8% confidence)

Query: 'Consumer complaint against seller for defective goods and refund'
 -> Predicted: CONSUMER (32.9% confidence)

Query: 'How to verify a digital signature certificate under Information Technology rules?'
 -> Predicted: CYBER (93.1% confidence)

Query: 'Admissibility of electronic records and secondary evidence under Bharatiya Sakshya Adhiniyam'
 -> Predicted: EVIDENCE (93.2% confidence)



### 8. Save model for backend API
We serialize the pipeline with `joblib` so that FastAPI can load it on startup to categorize user inquiries.

In [8]:
import joblib

out_path = data_dir / 'topic_clf.joblib'
joblib.dump(clf, out_path)
print(f'✓ Model saved to: {out_path}')
print('Ready for FastAPI backend.')

✓ Model saved to: backend\data\topic_clf.joblib
Ready for FastAPI backend.
